# Robust NLP Preprocessing Engine

## Project Description
A comprehensive text preprocessing pipeline for Natural Language Processing tasks that handles diverse text formats, noise, and edge cases while maintaining data integrity.

## Objectives
1. Implement a robust, production-grade text preprocessing function
2. Understand the impact of various preprocessing techniques
3. Handle edge cases and noisy real-world data
4. Analyze token patterns and text characteristics
5. Create a reusable pipeline for handling text collections

## Task 1: Conceptual Understanding

### Q1: Difference between "Love" and "love" in NLP
**Case Sensitivity Impact:**
- **Without lowercasing:** "Love" and "love" are treated as different tokens, causing vocabulary fragmentation and feature sparsity
- **With lowercasing:** Both become "love", consolidating semantic meaning and reducing dimensionality
- In sentiment analysis/text classification, this prevents models from learning redundant patterns

### Q2: Impact of NOT Removing Stopwords
**Consequences:**
- Increased dimensionality without semantic value (words like 'the', 'is', 'a' appear in almost every document)
- Noise in frequency analysis and word embeddings
- Higher computational cost and slower model training
- Diluted importance scores for meaningful words (TF-IDF, word frequency)
- Poor quality of topic modeling and clustering

### Q3: Two Real-World Cases Where Removing Stopwords is HARMFUL

**Case 1: Sentiment Analysis on Negations**
- Sentence: "This movie is **not** good"
- If we remove "not" (common stopword), we lose the negation context
- Model interprets it as "This movie is good" → Sentiment reversal
- Impact: Incorrect classification, poor model performance

**Case 2: Question Answering Systems**
- Query: "What **is** the capital **of** France?"
- Removing "is" and "of" destroys grammatical structure needed for parsing
- System cannot properly extract question intent
- Impact: Failed QA retrieval, wrong answers

### Q4: Stemming vs Lemmatization Comparison

| Aspect | Stemming | Lemmatization |
|--------|----------|---------------|
| **Definition** | Removes suffix/prefix (rule-based) | Converts to dictionary root form (knowledge-based) |
| **Example** | "running", "runs", "ran" → "run" | "running", "runs", "ran" → "run" |
| **Speed** | Very fast (simple rule matching) | Slower (requires morphological analysis) |
| **Accuracy** | May produce non-word stems | Always produces valid words |
| **Output Quality** | "organizational" → "organ" (incorrect) | "organizational" → "organize" (correct) |
| **Use Case** | Information retrieval, quick processing | NLP tasks requiring grammatical correctness |
| **Dependencies** | None | Requires POS tagging, vocabulary |
| **Production Use** | Search engines, indexing | Sentiment analysis, text classification |

## Task 2: Preprocessing Function Implementation

In [10]:
import re
import collections
from typing import Tuple, List, Dict

def preprocess_text(text: str) -> Tuple[List[str], str]:
    """
    Comprehensive text preprocessing function that handles multiple text anomalies.
    
    Args:
        text (str): Raw input text
    
    Returns:
        Tuple containing:
        - tokens (List[str]): List of processed tokens
        - cleaned_sentence (str): Cleaned sentence string
    """
    
    # Handle empty input
    if not text or not isinstance(text, str):
        return [], ""
    
    # Step 1: Lowercase conversion
    text = text.lower()
    
    # Step 2: Remove URLs (http, https, www patterns)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'www\S+', '', text)
    
    # Step 3: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Step 4: Remove emojis and special characters (keep only alphanumeric, spaces, hyphens)
    # This regex keeps: a-z, 0-9, spaces, and common punctuation for contractions
    text = re.sub(r'[^a-z0-9\s\-\'\.]', '', text)
    
    # Step 5: Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Step 6: Handle repeated characters (e.g., "soooo" → "so", "hellooooo" → "hello")
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # Step 7: Remove extra spaces (multiple spaces to single space)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 8: Tokenization (split by spaces)
    tokens = text.split()
    
    # Step 9: Remove words with length <= 2, except for "no" and "not" (important for negation)
    processed_tokens = []
    for token in tokens:
        if len(token) > 2 or token in ['no', 'not']:
            processed_tokens.append(token)
    
    # Create cleaned sentence from processed tokens
    cleaned_sentence = ' '.join(processed_tokens)
    
    return processed_tokens, cleaned_sentence

# Test the function
print("Function 'preprocess_text' successfully defined.")
print("\nSignature: preprocess_text(text: str) -> Tuple[List[str], str]")
print("Returns: (tokens, cleaned_sentence)")

Function 'preprocess_text' successfully defined.

Signature: preprocess_text(text: str) -> Tuple[List[str], str]
Returns: (tokens, cleaned_sentence)


## Task 3: Stress Testing with Diverse Sentences

In [11]:
# Define diverse test sentences covering edge cases
test_sentences = [
    "I love this product!!! 😍😍😍 Check it out: https://example.com",  # Emojis, URLs
    "Hello world! This is AMAZING!!!!! Contact me: john@email.com",      # Mixed case, email
    "The prices are $99 and €50... sooooo expensive!",                   # Numbers, repeated chars
    "LOL ROFL 😂 U r gr8! Visit www.coolsite.com 4 more",               # Slang, emojis, URLs
    "Machine Learning ML is NOT just hype... 2023-2024",                 # Acronyms, numbers, negation
    "This is absolutely BEAUTIFUL.... #amazing #love #bestday 🌟",       # Repeated chars, hashtags
    "123456 numbers only 789 @#$%^&*() special characters +++",          # Pure numbers, special chars
    "no way!!! This can't be true. Contact support@company.org NOW!!!",  # Negation words, email
    "Data science involves 95% math + 100% code 😎 https://AI.tech",    # Mixed elements
    "soooo funny... hahahaha.... I'm dying lol 💀💀💀 www.memes.com",   # Repeated, slang, emojis
    "NLP (Natural Language Processing) beats traditional methods!!!!!!!!" # Parentheses, repeated punctuation
]

print("="*80)
print("STRESS TESTING: DIVERSE SENTENCE PREPROCESSING")
print("="*80)

# Store results for later analysis
all_results = []

for idx, sentence in enumerate(test_sentences, 1):
    tokens, cleaned = preprocess_text(sentence)
    all_results.append({
        'original': sentence,
        'tokens': tokens,
        'cleaned': cleaned
    })
    
    print(f"\n[Sentence {idx}]")
    print(f"Original:        {sentence}")
    print(f"Tokens:          {tokens}")
    print(f"Cleaned Sent:    {cleaned}")

print("\n" + "="*80)
print(f"Total sentences processed: {len(test_sentences)}")
print("="*80)

STRESS TESTING: DIVERSE SENTENCE PREPROCESSING

[Sentence 1]
Original:        I love this product!!! 😍😍😍 Check it out: https://example.com
Tokens:          ['love', 'this', 'product', 'check', 'out']
Cleaned Sent:    love this product check out

[Sentence 2]
Original:        Hello world! This is AMAZING!!!!! Contact me: john@email.com
Tokens:          ['hello', 'world', 'this', 'amazing', 'contact']
Cleaned Sent:    hello world this amazing contact

[Sentence 3]
Original:        The prices are $99 and €50... sooooo expensive!
Tokens:          ['the', 'prices', 'are', 'and', 'expensive']
Cleaned Sent:    the prices are and expensive

[Sentence 4]
Original:        LOL ROFL 😂 U r gr8! Visit www.coolsite.com 4 more
Tokens:          ['lol', 'rofl', 'visit', 'more']
Cleaned Sent:    lol rofl visit more

[Sentence 5]
Original:        Machine Learning ML is NOT just hype... 2023-2024
Tokens:          ['machine', 'learning', 'not', 'just', 'hype.']
Cleaned Sent:    machine learning not just hyp

## Task 4: Token Analytics

In [12]:
def compute_token_analytics(results: List[Dict]) -> Dict:
    """
    Compute analytics for each sentence including token counts and lengths.
    
    Args:
        results: List of preprocessing results
    
    Returns:
        Dictionary with analytics for each sentence
    """
    analytics = []
    
    for idx, result in enumerate(results, 1):
        tokens = result['tokens']
        total_tokens = len(tokens)
        unique_tokens = len(set(tokens))
        
        # Calculate average token length
        avg_length = sum(len(token) for token in tokens) / total_tokens if total_tokens > 0 else 0
        
        # Calculate noise metric: 1 - (unique/total) - lower means more repeated words
        noise_score = 1 - (unique_tokens / total_tokens) if total_tokens > 0 else 0
        
        analytics.append({
            'sentence_idx': idx,
            'total_tokens': total_tokens,
            'unique_tokens': unique_tokens,
            'avg_length': round(avg_length, 2),
            'noise_score': round(noise_score, 3),
            'original': result['original'][:50] + '...' if len(result['original']) > 50 else result['original']
        })
    
    return analytics

# Compute analytics
analytics = compute_token_analytics(all_results)

print("\n" + "="*100)
print("TOKEN ANALYTICS FOR ALL SENTENCES")
print("="*100)
print(f"{'Idx':<5} {'Total':<8} {'Unique':<8} {'Avg Len':<10} {'Noise':<8} {'Sample Text'}")
print("-"*100)

for item in analytics:
    print(f"{item['sentence_idx']:<5} {item['total_tokens']:<8} {item['unique_tokens']:<8} "
          f"{item['avg_length']:<10} {item['noise_score']:<8} {item['original']}")

# Find sentences with most and least noise
most_noise_idx = max(range(len(analytics)), key=lambda i: analytics[i]['noise_score'])
least_noise_idx = min(range(len(analytics)), key=lambda i: analytics[i]['noise_score'])

print("\n" + "="*100)
print("KEY FINDINGS:")
print("="*100)
print(f"\nMost Noise: Sentence {analytics[most_noise_idx]['sentence_idx']}")
print(f"  - Noise Score: {analytics[most_noise_idx]['noise_score']}")
print(f"  - Reason: High word repetition (many duplicate tokens)")
print(f"  - Original: {most_noise_idx + 1} tokens, {analytics[most_noise_idx]['unique_tokens']} unique")

print(f"\nMost Meaningful: Sentence {analytics[least_noise_idx]['sentence_idx']}")
print(f"  - Noise Score: {analytics[least_noise_idx]['noise_score']}")
print(f"  - Reason: All tokens are unique or mostly unique (rich vocabulary)")
print(f"  - Original: {all_results[least_noise_idx]['tokens']} tokens, {analytics[least_noise_idx]['unique_tokens']} unique")


TOKEN ANALYTICS FOR ALL SENTENCES
Idx   Total    Unique   Avg Len    Noise    Sample Text
----------------------------------------------------------------------------------------------------
1     5        5        4.6        0.0      I love this product!!! 😍😍😍 Check it out: https://e...
2     5        5        5.6        0.0      Hello world! This is AMAZING!!!!! Contact me: john...
3     5        5        4.8        0.0      The prices are $99 and €50... sooooo expensive!
4     4        4        4.0        0.0      LOL ROFL 😂 U r gr8! Visit www.coolsite.com 4 more
5     5        5        5.4        0.0      Machine Learning ML is NOT just hype... 2023-2024
6     6        6        7.0        0.0      This is absolutely BEAUTIFUL.... #amazing #love #b...
7     4        4        7.0        0.0      123456 numbers only 789 @#$%^&*() special characte...
8     7        7        4.14       0.0      no way!!! This can't be true. Contact support@comp...
9     5        5        5.4        0.0

## Task 5: Frequency Analysis

In [13]:
# Combine all tokens from all sentences
all_tokens = []
for result in all_results:
    all_tokens.extend(result['tokens'])

print(f"Total tokens across all sentences: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}")

# Use collections.Counter for frequency analysis
token_frequency = collections.Counter(all_tokens)

print("\n" + "="*80)
print("TOP 10 MOST FREQUENT WORDS")
print("="*80)
for rank, (token, count) in enumerate(token_frequency.most_common(10), 1):
    percentage = (count / len(all_tokens)) * 100
    print(f"{rank:2d}. '{token:<15}' : {count:3d} occurrences ({percentage:5.2f}%)")

print("\n" + "="*80)
print("TOP 5 LEAST FREQUENT WORDS (appearing once)")
print("="*80)

# Get least frequent words (appearing only once)
least_frequent = [(token, count) for token, count in token_frequency.items() if count == 1]
least_frequent_sorted = sorted(least_frequent, key=lambda x: x[0])  # Sort alphabetically

if least_frequent_sorted:
    print(f"Total unique words appearing once: {len(least_frequent_sorted)}")
    print("\nFirst 5 (alphabetically):")
    for rank, (token, count) in enumerate(least_frequent_sorted[:5], 1):
        print(f"{rank}. '{token}'")
else:
    print("No words appear exactly once.")

# Vocabulary statistics
print("\n" + "="*80)
print("VOCABULARY STATISTICS")
print("="*80)
print(f"Total word occurrences: {len(all_tokens)}")
print(f"Unique words: {len(set(all_tokens))}")
print(f"Type-Token Ratio (TTR): {len(set(all_tokens))/len(all_tokens):.3f}")
print(f"Average word frequency: {len(all_tokens)/len(set(all_tokens)):.2f}")

Total tokens across all sentences: 58
Unique tokens: 51

TOP 10 MOST FREQUENT WORDS
 1. 'this           ' :   4 occurrences ( 6.90%)
 2. 'love           ' :   2 occurrences ( 3.45%)
 3. 'amazing        ' :   2 occurrences ( 3.45%)
 4. 'contact        ' :   2 occurrences ( 3.45%)
 5. 'lol            ' :   2 occurrences ( 3.45%)
 6. 'product        ' :   1 occurrences ( 1.72%)
 7. 'check          ' :   1 occurrences ( 1.72%)
 8. 'out            ' :   1 occurrences ( 1.72%)
 9. 'hello          ' :   1 occurrences ( 1.72%)
10. 'world          ' :   1 occurrences ( 1.72%)

TOP 5 LEAST FREQUENT WORDS (appearing once)
Total unique words appearing once: 46

First 5 (alphabetically):
1. 'absolutely'
2. 'and'
3. 'are'
4. 'beats'
5. 'beautiful.'

VOCABULARY STATISTICS
Total word occurrences: 58
Unique words: 51
Type-Token Ratio (TTR): 0.879
Average word frequency: 1.14


## Task 6: Full Pipeline Implementation

In [14]:
def full_pipeline(text_list: List[str]) -> Dict:
    """
    Process multiple texts through the complete preprocessing pipeline.
    
    Args:
        text_list: List of raw text strings
    
    Returns:
        Dictionary containing:
        - 'tokens': Combined list of all tokens from all texts
        - 'clean_sentences': List of cleaned sentences
        - 'token_per_sentence': List of token lists for each sentence
        - 'summary': Processing statistics
    """
    if not text_list:
        return {"tokens": [], "clean_sentences": [], "token_per_sentence": [], "summary": {}}
    
    all_tokens_combined = []
    clean_sentences = []
    token_per_sentence = []
    
    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens_combined.extend(tokens)
        clean_sentences.append(cleaned)
        token_per_sentence.append(tokens)
    
    # Generate summary statistics
    summary = {
        'total_texts': len(text_list),
        'total_tokens': len(all_tokens_combined),
        'unique_tokens': len(set(all_tokens_combined)),
        'avg_tokens_per_sentence': round(len(all_tokens_combined) / len(text_list), 2) if text_list else 0,
        'texts_with_no_tokens': sum(1 for t in token_per_sentence if len(t) == 0)
    }
    
    return {
        "tokens": all_tokens_combined,
        "clean_sentences": clean_sentences,
        "token_per_sentence": token_per_sentence,
        "summary": summary
    }

# Test the full pipeline
print("Testing Full Pipeline with our test sentences...\n")
pipeline_result = full_pipeline(test_sentences)

print("="*80)
print("FULL PIPELINE RESULTS")
print("="*80)
print(f"\nSummary Statistics:")
for key, value in pipeline_result['summary'].items():
    print(f"  {key}: {value}")

print(f"\nTotal tokens extracted: {len(pipeline_result['tokens'])}")
print(f"Total clean sentences: {len(pipeline_result['clean_sentences'])}")
print(f"\nFirst 3 clean sentences:")
for idx, sentence in enumerate(pipeline_result['clean_sentences'][:3], 1):
    print(f"  {idx}. {sentence}")

Testing Full Pipeline with our test sentences...

FULL PIPELINE RESULTS

Summary Statistics:
  total_texts: 11
  total_tokens: 58
  unique_tokens: 51
  avg_tokens_per_sentence: 5.27
  texts_with_no_tokens: 0

Total tokens extracted: 58
Total clean sentences: 11

First 3 clean sentences:
  1. love this product check out
  2. hello world this amazing contact
  3. the prices are and expensive


## Task 7: Error Handling and Edge Cases

In [15]:
# Define edge case test scenarios
edge_cases = [
    ("", "Empty string"),
    ("   ", "Only whitespace"),
    ("😍😂🎉🌟💯", "Only emojis"),
    ("12345 67890 98765", "Only numbers"),
    ("@#$%^&*()!!", "Only special characters"),
    ("a b c", "Single character words only"),
    ("no not", "Only excluded words (no/not)"),
    (None, "None input"),
    (123, "Non-string input (integer)"),
    ("...   ...   ...", "Only punctuation and spaces")
]

print("="*100)
print("ERROR HANDLING & EDGE CASE TESTING")
print("="*100)

for test_input, description in edge_cases:
    print(f"\n[Test Case] {description}")
    print(f"Input: {repr(test_input)}")
    
    try:
        tokens, cleaned = preprocess_text(test_input)
        print(f"✓ Success")
        print(f"  Tokens: {tokens}")
        print(f"  Cleaned: '{cleaned}'")
        print(f"  Token count: {len(tokens)}")
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {str(e)}")

print("\n" + "="*100)
print("ERROR HANDLING SUMMARY")
print("="*100)
print("""
The preprocessing function handles edge cases gracefully:

1. Empty Strings: Returns empty tokens and empty cleaned sentence
2. Only Emojis: Removed by character filtering, returns []
3. Only Numbers: Removed by number filtering, returns []
4. Special Characters: Removed by character filtering, returns []
5. None/Invalid Types: Checked with isinstance(), returns empty safely
6. Whitespace Only: Stripped and returns empty after tokenization
7. Single Characters: Filtered out (length <= 2 rule), except 'no' and 'not'

This ensures the function never crashes on malformed input.
""")

ERROR HANDLING & EDGE CASE TESTING

[Test Case] Empty string
Input: ''
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only whitespace
Input: '   '
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only emojis
Input: '😍😂🎉🌟💯'
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only numbers
Input: '12345 67890 98765'
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only special characters
Input: '@#$%^&*()!!'
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Single character words only
Input: 'a b c'
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only excluded words (no/not)
Input: 'no not'
✓ Success
  Tokens: ['no', 'not']
  Cleaned: 'no not'
  Token count: 2

[Test Case] None input
Input: None
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Non-string input (integer)
Input: 123
✓ Success
  Tokens: []
  Cleaned: ''
  Token count: 0

[Test Case] Only punctuation an